In [0]:
catalog = "workspace"
schema = "retail"

customers_bronze_table = f"{catalog}.{schema}.customers_bronze"
products_bronze_table = f"{catalog}.{schema}.products_bronze"
orders_bronze_table = f"{catalog}.{schema}.orders_bronze"

customers_silver_table = f"{catalog}.{schema}.customers_silver"
products_silver_table = f"{catalog}.{schema}.products_silver"
orders_silver_table = f"{catalog}.{schema}.orders_silver"

In [0]:
customers_bronze_df = spark.table(customers_bronze_table)
products_bronze_df = spark.table(products_bronze_table)
orders_bronze_df = spark.table(orders_bronze_table)

In [0]:
from pyspark.sql import functions as F

duplicate_customer_ids_df = (
    customers_bronze_df
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)
duplicate_customer_id_count = duplicate_customer_ids_df.count()

In [0]:
invalid_email_count = (
    customers_bronze_df
    .filter(
        F.col("email").isNull()
        | (F.trim(F.col("email")) == "")
    )
    .count()
)


In [0]:
future_registration_date_count = (
    customers_bronze_df
    .filter(F.col("registration_date") > F.current_date())
    .count()
)

In [0]:
invalid_registration_date_count = (
    customers_bronze_df
    .filter(
        F.col("registration_date").isNull()
        | (F.col("registration_date") > F.current_date())
    )
    .count()
)

In [0]:
print("Customer data-quality checks")
print("--------------------------------")
print(f"Dubbele customer_id's: {duplicate_customer_id_count}")
print(f"Lege e-mailadressen: {invalid_email_count}")
print(f"Ongeldige registratiedatums: {invalid_registration_date_count}")

In [0]:
customers_silver_df = (
    customers_bronze_df
    .filter(
        F.col("email").isNotNull()
        & (F.trim(F.col("email")) != "")
    )
    .filter(
        F.col("registration_date").isNotNull()
        & (F.col("registration_date") <= F.current_date())
    )
    .dropDuplicates(["customer_id"])
)

In [0]:
bronze_customer_count = customers_bronze_df.count()
silver_customer_count = customers_silver_df.count()

print(f"Bronze records: {bronze_customer_count}")
print(f"Silver records: {silver_customer_count}")
print(f"Verwijderde records: {bronze_customer_count - silver_customer_count}")

In [0]:
customers_silver_df.show(5)

In [0]:
(
    customers_silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(customers_silver_table)
)

In [0]:
customers_silver_df = spark.table(customers_silver_table)
customers_silver_df.show(5)

In [0]:
duplicate_product_ids_df = (
    products_bronze_df
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
)
duplicate_product_ids_count = duplicate_product_ids_df.count()

In [0]:
invalid_product_price_count = (
    products_bronze_df
    .filter(
        (F.col("unit_price").isNull())
        | (F.col("unit_price") <= 0)
    )
    .count()
)

In [0]:
print("Product data-quality checks")
print("--------------------------------")
print(f"Dubbele product_id's: {duplicate_product_ids_count}")
print(f"Ongeldige prijzen: {invalid_product_price_count}")

In [0]:
products_silver_df = (
    products_bronze_df
    .filter(
        (F.col("unit_price").isNotNull())
        & (F.col("unit_price") > 0)
    )
    .dropDuplicates(["product_id"])
)

In [0]:
bronze_product_count = products_bronze_df.count()
silver_product_count = products_silver_df.count()

print(f"Bronze records: {bronze_product_count}")
print(f"Silver records: {silver_product_count}")
print(f"Verwijderde records: {bronze_product_count - silver_product_count}")

In [0]:
products_silver_df.show(5)

In [0]:
(
    products_silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(products_silver_table)
)

In [0]:
products_silver_df = spark.table(products_silver_table)
products_silver_df.show(5)

In [0]:
invalid_order_discount_count = (
    orders_bronze_df
    .filter(
        (F.col("discount") < 0 )
        | (F.col("discount") > 1)
    )
    .count()
)

invalid_order_quantity_count = (
    orders_bronze_df
    .filter(
        (F.col("quantity") <= 0 )
    )
    .count()
)

invalid_order_revenue_count = (
    orders_bronze_df
    .filter(
        F.abs(
            (
                F.col("unit_price")
                * (1 - F.col("discount"))
                * F.col("quantity")
            )
            - F.col("revenue")
        ) > 0.01
    )
    .count()
)


In [0]:
print("Order data-quality checks")
print("--------------------------------")
print(f"Ongeldige korting: {invalid_order_discount_count}")
print(f"Ongeldige hoeveelheid: {invalid_order_quantity_count}")
print(f"Ongeldige omzet: {invalid_order_revenue_count}")



In [0]:
orders_silver_df = (
    orders_bronze_df
    .filter(
        (F.col("discount") >= 0)
        & (F.col("discount") <= 1)
        & (F.col("quantity") > 0)
        & (
            F.abs(
                (
                    F.col("unit_price")
                    * (1 - F.col("discount"))
                    * F.col("quantity")
                )
                - F.col("revenue")
            ) <= 0.01
        )
    )
    .dropDuplicates(["order_id"])
)

In [0]:
bronze_order_count = orders_bronze_df.count()
silver_order_count = orders_silver_df.count()

print(f"Bronze records: {bronze_order_count}")
print(f"Silver records: {silver_order_count}")
print(f"Verwijderde records: {bronze_order_count - silver_order_count}")

In [0]:
(
    orders_silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(orders_silver_table)
)

In [0]:
orders_silver_df = spark.table(orders_silver_table)
orders_silver_df.show(5)